In [ ]:
reset

In [1]:
import os
import sys
# block warnings from printing
import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

import collections
import pandas as pd
import xarray as xr
import cf_xarray as cf
xr.set_options(keep_attrs=True)
import netCDF4 as nc
import numpy as np
np.seterr(divide='ignore', invalid='ignore')
import metpy.calc as mp
from metpy.units import units
from scipy.stats import ttest_ind, ttest_rel
from datetime import datetime

import cartopy
cartopy.config['data_dir'] = "/discover/nobackup/projects/jh_tutorials/JH_examples/JH_datafiles/Cartopy"
cartopy.config['pre_existing_data_dir'] = "/discover/nobackup/projects/jh_tutorials/JH_examples/JH_datafiles/Cartopy"
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from shapely.geometry.polygon import LinearRing

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.offsetbox import AnchoredText
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
from matplotlib.colors import TwoSlopeNorm
from matplotlib import cm
from matplotlib.colors import ListedColormap,LinearSegmentedColormap
import cmocean.cm as cmo
import seaborn as sns

# settings
%config InlineBackend.figure_format = 'retina'

# add path to custom functions
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path+"/py_functions")
from map_plot_tools import *
from colorbar_funcs import *
from data_funcs import *
from stats_funcs import *

In [ ]:
### +++ DATA PATHS +++ ###

# product keys
keys=['obs','flor']
obs_prods=['merra2']
flor_runs=['ctrl','hitopo'] #['ctrl','hicam','hitopo']
flor_simNames=['ctrl','hitopo'] #['ctrl','cam','hitopo']

# store file paths in dictionary
dpath0='/discover/nobackup/projects/giss/baldwin_nip/dmkumar' # top level data directory
opath='/home/dmkumar/JupyterLinks/notebooks/topo_nam/figs'
files={ 'obs'   : { 'merra2' : {} },
        'flor'  : { 'ctrl'   : {},
                   #'hicam'  : {}, 
                   'hitopo' : {} },
        '2xco2' : { 'ctrl'   : {},
                   #'hicam'  : {}, 
                   'hitopo' : {} },
      }

for key in ['obs']:
    for i,run in enumerate(obs_prods):
        files[key][run]['h500'] = f'{dpath0}/obs_data/merra2/merra2.H.1980-2022.monthly.nc' 
        files[key][run]['h700'] = f'{dpath0}/obs_data/merra2/merra2.H.1980-2022.monthly.nc' 

for key in ['flor']:
    for i,run in enumerate(flor_runs):
        simName=flor_simNames[i]
        for fvarn in ['h500','h700']: 
            files[key][run][fvarn] = f'{dpath0}/FLOR/{run}/pi/flor.{simName}.{fvarn}.monthly.nc' 

for key in ['2xco2']:
    for i,run in enumerate(flor_runs):
        simName=flor_simNames[i]
        for fvarn in ['h500','h700']: 
            files[key][run][fvarn] = f'{dpath0}/FLOR/{run}/2xco2/flor.{simName}.2xco2.{fvarn}.monthly.nc' 

for key in ['topo']:
    files[key] = {}
    files[key]['etopo'] = f'{dpath0}/topo_files/obs.etopo5.zsurf.nc'
    files[key]['ctrl'] = f'{dpath0}/topo_files/flor.ctrl.zsurf.nc'
    files[key]['hicam'] = f'{dpath0}/topo_files/flor.cam.zsurf.nc'
    files[key]['hitopo'] = f'{dpath0}/topo_files/flor.hitopo.zsurf.nc'

In [ ]:
### +++ ORGANIZE DATA +++ ###
# diagnostics
varns = ['h500'] #,'h700']
swps = ['swp500'] #,'swp700']
# pressure level
plevs = [500,700]
# time bounds
n_years=50
n_keep_days=-1 * (n_years * 365) # number of time steps to keep in days
n_keep_mons=-1 * (n_years * 12)  # number of time steps to keep in months

# initialize dictionaries
dat={ 'obs'   : { 'merra2' : {} },
      'pi'    : { 'ctrl'   : {},
                  #'hicam'  : {}, 
                  'hitopo' : {} } }#,

      '2xco2' : { 'ctrl'   : {},
                  #'hicam'  : {}, 
                  'hitopo' : {} },
      }

print('Working on...')

for key in ['obs']:
    print(f'{key}')
    for run in obs_prods:
        for i,ovarn in enumerate(['H','H']):
            varn=varns[i]
            plev=plevs[i]
            ds = xr.open_dataset(files[key][run][varn])[ovarn].sel(lev=plev).squeeze(drop=True) # select specific pressure level
            ds_flip = lonFlip(ds) # switch lons from -180:180 to 0:360
            dat[key][run][varn] = ds_flip
            del ds
            del ds_flip
            swp = swps[i]
            dat[key][run][swp] = dat[key][run][varn] - dat[key][run][varn].mean(dim='lon')
            
for key in ['pi']: #,'2xco2']:
    print(f'{key}')
    for run in flor_runs:
        for i,varn in enumerate(varns):
            ds = xr.open_dataset(files[key][run][varn])[varn][n_keep_mons:,:,:] # keep only last 50 years 
            ds = ds.rename({'grid_xt':'lon','grid_yt':'lat'}) # update coordinate names to match merra2
            dat[key][run][varn] = ds
            del ds
            swp = swps[i]
            dat[key][run][swp] = dat[key][run][varn] - dat[key][run][varn].mean(dim='lon')

print('Done.')


topo = {}
print('Loading surface height data.')
for key in ['topo']:
    for case in ['etopo']:
        ds = xr.open_dataset(files[key][case]).ROSE.rename({'ETOPO05_X':'lon', 'ETOPO05_Y':'lat'})
        topo[case] = ds.where(ds>0, np.nan)
        del ds
    for case in flor_runs:
        ds = xr.open_dataset(files[key][case]).ZSURF.rename({'GRID_XT':'lon', 'GRID_YT':'lat'})
        topo[case] = ds.where(ds>0, np.nan)
        del ds
print('Done.')

In [ ]:
### +++ CALCULATE TIME-MEANS +++ ###
season='JAS'
mons=[7,8,9]
varns=['h500','swp500'] #,'h700','swp700']

jas_mean = { 'obs'  : { 'merra2':{} },
             'pi'  : { 'ctrl':{}, 'hitopo':{} } } #,
             '2xco2' : { 'ctrl':{}, 'hitopo':{} } }
ann_jas_mean = { 'obs'  : { 'merra2':{} },
                 'pi'  : { 'ctrl':{}, 'hitopo':{} } } #,
                 '2xco2' : { 'ctrl':{}, 'hitopo':{} } }

print('Calculating seasonal means for...')
for key in ['obs']:
    print(f'{key}')
    for run in obs_prods:
        for varn in varns:
            # seasonal mean for whole timeseries
            custom_seasons = xr.where(dat[key][run][varn]['time'].dt.month.isin(mons), season, 'Other')
            jas_mean[key][run][varn] = dat[key][run][varn].groupby(custom_seasons).mean('time').rename({'month':'season'}).sel(season=season)
            # seasonal mean by year
            ann_jas_mean[key][run][varn] = dat[key][run][varn].sel(time=dat[key][run][varn]['time'].dt.month.isin(mons)).groupby('time.year').mean(dim='time')
      
for key in ['pi','2xco2']:
    print(f'{key}')
    for run in flor_runs:
        for varn in varns:
            # seasonal mean for whole timeseries
            custom_seasons = xr.where(dat[key][run][varn]['time'].dt.month.isin(mons), season, 'Other')
            jas_mean[key][run][varn] = dat[key][run][varn].groupby(custom_seasons).mean('time').rename({'month':'season'}).sel(season=season)
            # seasonal mean by year
            ann_jas_mean[key][run][varn] = dat[key][run][varn].sel(time=dat[key][run][varn]['time'].dt.month.isin(mons)).groupby('time.year').mean(dim='time')

print('Done.')

In [ ]:
### +++ COMPARING ALL MODEL RUNS TO OBS. +++ ###

## initialize dictionaries
# for re-gridded obs data
jas_mean_regrid = { 'obs_flor' : {} }
ann_jas_mean_regrid = { 'obs_flor' : {} }
# for significance testing results
obs_diff      = { 'pi'  : { 'ctrl':{}, 'hitopo':{} } }

# First, need to put obs data on same grid as model output
regrid_keys = ['obs_flor']

print('Re-gridding obs.')
for i,key in enumerate(['pi']):
    lats=dat[key]['ctrl']['h500'].lat
    lons=dat[key]['ctrl']['h500'].lon
    regrid_key=regrid_keys[i]
    for varn in varns:
        # interpolate obs to CM2.5-FLOR grid
        jas_mean_regrid[regrid_key][varn] = jas_mean['obs']['merra2'][varn].interp(lat=lats, lon=lons, method='linear')
        ann_jas_mean_regrid[regrid_key][varn] = ann_jas_mean['obs']['merra2'][varn].interp(lat=lats, lon=lons, method='linear')
print('Done.\n')

# Determine statistical significance of obs-model differences based on students t-test
print('Significance testing for:')
for key in obs_diff.keys():
    print(f'{key}')
    for run in flor_runs:
        print(f'...{run}')
        for varn in varns:
            obs_diff[key][run][varn] = jas_mean_regrid['obs_flor'][varn] - jas_mean[key][run][varn]
print('Done.')

In [ ]:
"""
### +++ COMPARING ALL MODEL RUNS TO OBS. +++ ###

## initialize dictionaries
# for re-gridded obs data
jas_mean_regrid = { 'obs_flor' : {} }
ann_jas_mean_regrid = { 'obs_flor' : {} }
# for significance testing results
obs_diff      = { 'flor'  : { 'ctrl':{}, 'hitopo':{} },
                  '2xco2' : { 'ctrl':{}, 'hitopo':{} } }
obs_diff_mask = { 'flor'  : { 'ctrl':{}, 'hitopo':{} },
                  '2xco2' : { 'ctrl':{}, 'hitopo':{} }  }
obs_ptvals    = { 'flor'  : { 'ctrl':{}, 'hitopo':{} },
                  '2xco2' : { 'ctrl':{}, 'hitopo':{} }  }

# First, need to put obs data on same grid as model output
regrid_keys = ['obs_flor']

print('Re-gridding obs.')
for i,key in enumerate(['flor']):
    lats=dat[key]['ctrl']['h500'].lat
    lons=dat[key]['ctrl']['h500'].lon
    regrid_key=regrid_keys[i]
    for varn in varns:
        # interpolate obs to CM2.5-FLOR grid
        jas_mean_regrid[regrid_key][varn] = jas_mean['obs']['merra2'][varn].interp(lat=lats, lon=lons, method='linear')
        ann_jas_mean_regrid[regrid_key][varn] = ann_jas_mean['obs']['merra2'][varn].interp(lat=lats, lon=lons, method='linear')
print('Done.\n')

# Determine statistical significance of obs-model differences based on students t-test
print('Significance testing for:')
for key in obs_diff.keys():
    print(f'{key}')
    for run in flor_runs:
        print(f'...{run}')
        for varn in varns:
            diff_, diff_mask_, ptvals_ = sigtest2n(ann_jas_mean_regrid['obs_flor'][varn], ann_jas_mean[key][run][varn], 
                                                   jas_mean_regrid['obs_flor'][varn], jas_mean[key][run][varn])
            obs_diff[key][run][varn] = diff_
            obs_diff_mask[key][run][varn] = diff_mask_
            obs_ptvals[key][run][varn] = ptvals_
print('Done.')
"""

In [ ]:
### +++ COMPARING ALL MODIFED TOPO MODEL RUNS TO MODEL CTRL +++ ###

# initialize dictionaries
model_diff      = { 'pi'  : { 'hitopo':{} },
                    '2xco2' : { 'hitopo':{} } }
model_diff_mask = { 'pi'  : { 'hitopo':{} },
                    '2xco2' : { 'hitopo':{} } }
model_ptvals    = { 'pi'  : { 'hitopo':{} },
                    '2xco2' : { 'hitopo':{} }}
# names of modified topography runs
flor_mod_runs=['hitopo']

print('Significance testing for:')
for key in model_diff.keys():
    print(f'{key}')
    for run in flor_mod_runs:
        print(f'...{run}')
        for varn in varns:
            # calculate significance of model - ctrl difference
            diff_, diff_mask_, ptvals_ = sigtest2n(ann_jas_mean[key][run][varn], ann_jas_mean[key]['ctrl'][varn],
                                                   jas_mean[key][run][varn], jas_mean[key]['ctrl'][varn])
            model_diff[key][run][varn] = diff_
            model_diff_mask[key][run][varn] = diff_mask_
            model_ptvals[key][run][varn] = ptvals_

print('Done.')

In [ ]:
### +++ COMPARING ALL MODEL 2XCO2 TO MODEL PI +++ ###

# initialize dictionaries
future_diff      = { '2xco2' : {'ctrl'   : {},
                                'hitopo' : {} }
                   }
future_diff_mask = { '2xco2' : {'ctrl'   : {},
                                'hitopo' : {} }
                   }
future_ptvals    = { '2xco2' : {'ctrl'   : {},
                                'hitopo' : {} }
                   }

print('Significance testing for:')
for key in ['2xco2']:
    print(f'{key}')
    for run in flor_runs:
        print(f'...{run}')
        for varn in varns:
            # calculate significance of model - ctrl difference
            diff_, diff_mask_, ptvals_ = sigtest2n(ann_jas_mean[key][run][varn], ann_jas_mean['pi'][run][varn],
                                                   jas_mean[key][run][varn], jas_mean['pi'][run][varn])
            future_diff[key][run][varn] = diff_
            future_diff_mask[key][run][varn] = diff_mask_
            future_ptvals[key][run][varn] = ptvals_

print('Done.')

In [ ]:
### +++ BIAS IMPROVEMENT +++ ###

bias_change        = { 'flor'  : { 'hitopo':{} },
                       '2xco2' : { 'hitopo':{} } }
bias_change_masked = { 'flor'  : { 'hitopo':{} },
                       '2xco2' : { 'hitopo':{} } }

# calculate difference in the absolute value of the model-obs precip difference
# to determine whether or not the change in precipitation is a reduction in the ctrl model bias
for key in bias_change.keys():
    for run in flor_mod_runs:
        for varn in varns:
            bias_change[key][run][varn] = np.abs(obs_diff[key][run][varn])-np.abs(obs_diff[key]['ctrl'][varn])
            bias_change_masked[key][run][varn] = bias_change[key][run][varn].where(model_diff_mask[key][run][varn].mask==False,np.nan)

In [ ]:
### +++ BIAS IMPROVEMENT +++ ###

bias_change        = { 'pi'  : { 'ctrl':{}, 'hitopo':{} },
                       '2xco2' : { 'ctrl':{}, 'hitopo':{} } }
bias_change_masked = { 'pi'  : { 'ctrl':{}, 'hitopo':{} },
                       '2xco2' : { 'ctrl':{}, 'hitopo':{} } }

# calculate difference in the absolute value of the model-obs precip difference
# to determine whether or not the change in precipitation is a reduction in the ctrl model bias
varn='h500'
for key in bias_change.keys():
    for run in flor_runs:
        bias_change[key][run][varn] = np.abs(obs_diff[key][run][varn])-np.abs(obs_diff['pi']['ctrl'][varn]) # comparing change in bias explicitly to FLOR PI CTRL
        if run=='hitopo':
            bias_change_masked[key][run][varn] = bias_change[key][run][varn].where(model_diff_mask[key][run][varn].mask==False,np.nan) # only keep bias info where there is a statistically significant change in the field
        else:
            if key=='2xco2':
                bias_change_masked[key][run][varn] = bias_change[key][run][varn].where(future_diff_mask[key][run][varn].mask==False,np.nan)
            else:
                bias_change_masked[key][run][varn] = {}

## FIGURES

### h500 climo and stationary wave pattern

In [ ]:
# -------------------- #
#       Settings       #
# -------------------- #
# plot specs
lw=1
text_kw={'color':'k', 'weight':'bold', 'size':20, 'ha':'center', 'va':'bottom'}
text_kw2={'color':'k', 'weight':'bold', 'size':26, 'ha':'center', 'va':'center'}
text_kw3={'color':'k', 'weight':'normal', 'size':14, 'ha':'left', 'va':'center'}
titles=np.array(['MERRA-2','CTRL',r'HI$\mathbf{_{GBL}}$',
                 '','',''])
letters=['A','B','C','D','E','F']
tx=-105
ty=66
# var specs
lon = jas_mean['pi']['ctrl']['h500'].lon
lat = jas_mean['pi']['ctrl']['h500'].lat
# topography contours
zlevels=np.linspace(1000,4000,8)
# climo colormap
cmap=cm.RdYlBu_r
vmin=5500
vmax=6000
levels=np.linspace(vmin, vmax, 21)
norm=mpl.colors.BoundaryNorm(levels, cmap.N)
hlevels=np.arange(vmin-50, vmax+50, 25)
# bias colormap
dcmap=cm.RdBu_r
dvmin=-60
dvmax=60
dlevels=np.linspace(dvmin, dvmax, 21)
dnorm=mpl.colors.BoundaryNorm(dlevels, dcmap.N)
# map specs
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[225,290,10,65]
cols=[0,2]

fig, ax = plt.subplots(nrows=2, ncols=3, figsize=(20,12), layout='constrained', subplot_kw={'projection':proj})

for i,title in enumerate(['MERRA-2', 'CTRL', r'HI$\mathbf{_{GBL}}$']):
    ax[0,i].text(tx, ty, title, **text_kw)
    
#=== OBS ===
mlon = jas_mean['obs']['merra2']['h500'].lon
mlat = jas_mean['obs']['merra2']['h500'].lat
ax[0,0].pcolormesh(mlon, mlat, jas_mean['obs']['merra2']['h500'].squeeze(), cmap=cmap, norm=norm, transform=trans)
#ax[0,0].contour(mlon, mlat, jas_mean['obs']['merra2']['h500'], colors='k', levels=hlevels, linewidth=1.5, transform=trans)
ax[0,0].contour(topo['etopo'].lon, topo['etopo'].lat, topo['etopo'], levels=np.linspace(1000,4000,4), linewidths=1, colors='black', transform=trans)

ax[1,0].pcolormesh(mlon, mlat, jas_mean['obs']['merra2']['swp500'].squeeze(), cmap=dcmap, norm=dnorm, transform=trans)
ax[1,0].contour(topo['etopo'].lon, topo['etopo'].lat, topo['etopo'], levels=np.linspace(1000,4000,4), linewidths=1, colors='black', transform=trans)

#=== FLOR ===
ax[0,1].pcolormesh(lon, lat, jas_mean['pi']['ctrl']['h500'], cmap=cmap, norm=norm, transform=trans)
cf=ax[0,2].pcolormesh(lon, lat, jas_mean['pi']['hitopo']['h500'], cmap=cmap, norm=norm, transform=trans)

ax[1,1].pcolormesh(lon, lat, jas_mean['pi']['ctrl']['swp500'], cmap=dcmap, norm=dnorm, transform=trans)
cf2=ax[1,2].pcolormesh(lon, lat, jas_mean['pi']['hitopo']['swp500'], cmap=dcmap, norm=dnorm, transform=trans)

for i,run in enumerate(['ctrl','hitopo']):
    ax[0,i+1].contour(lon, lat, topo[run], levels=zlevels, linewidths=1, colors='black', transform=trans)
    ax[1,i+1].contour(lon, lat, topo[run], levels=zlevels, linewidths=1, colors='black', transform=trans)
    
for i, ax in enumerate(ax.flat): 
    # subplot labels
    ax.text(-136, ty+1, letters[i], **text_kw2)
    # map properties
    ax.coastlines(color='k', linewidth=1.5)
    ax.set_extent(map_bnds, crs=trans)
    gl=ax.gridlines(crs=trans, lw=.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
    gl.bottom_labels=True; gl.left_labels=True; gl.top_labels=False; gl.right_labels=False
    gl.xformatter=LONGITUDE_FORMATTER
    gl.yformatter=LATITUDE_FORMATTER
    gl.xlabel_style={'color': 'black', 'weight': 'normal', 'size':14}
    gl.ylabel_style={'color': 'black', 'weight': 'normal', 'size':14}

# climo
cax=fig.add_axes([1, .55, 0.02, 0.4])
cbar=fig.colorbar(cf, ticks=np.linspace(5500,6000,11), orientation='vertical', extend='both', cax=cax) 
cbar.set_label('500 hPa Geopotential Height [m]', size=18, ha='center')
cbar.ax.tick_params(labelsize=18)

# bias
cax=fig.add_axes([1, .05, 0.02, 0.4])
cbar=fig.colorbar(cf2, ticks=np.linspace(-60,60,11), orientation='vertical', extend='both', cax=cax) 
cbar.set_label('Stationary Wave Pattern [m]', size=18, ha='center')
cbar.ax.tick_params(labelsize=18)

#plt.savefig(f'figs/h500.swp.flor.pi.climo.pdf', transparent=False, bbox_inches='tight')
#plt.savefig(f'figs/h500.swp.flor.pi.climo.png', transparent=False, bbox_inches='tight')

In [ ]:
# -------------------- #
#       Settings       #
# -------------------- #
# plot specs
lw=1
text_kw={'color':'k', 'weight':'bold', 'size':20, 'ha':'center', 'va':'bottom'}
text_kw2={'color':'k', 'weight':'bold', 'size':26, 'ha':'center', 'va':'center'}
text_kw3={'color':'k', 'weight':'normal', 'size':14, 'ha':'left', 'va':'center'}
titles=np.array(['MERRA-2','CTRL',r'HI$\mathbf{_{GBL}}$',
                 '','',''])
letters=['A','B','C','D','E','F','G','H']
tx=-105
ty=66
# var specs
lon = jas_mean['pi']['ctrl']['h500'].lon
lat = jas_mean['pi']['ctrl']['h500'].lat
# topography contours
zlevels=np.linspace(1000,4000,8)
# h500 colormap
cmap=cm.RdYlBu_r
vmin=5500
vmax=6000
levels=np.linspace(vmin, vmax, 21)
norm=mpl.colors.BoundaryNorm(levels, cmap.N)
hlevels=np.arange(vmin-50, vmax+50, 25)
# swp500 colormap
dcmap=cm.RdBu_r
dvmin=-60
dvmax=60
dlevels=np.linspace(dvmin, dvmax, 21)
dnorm=mpl.colors.BoundaryNorm(dlevels, dcmap.N)
# diff colormap
dcmap2=cmo.balance
# map specs
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[225,290,10,65]
cols=[0,2]

fig, ax = plt.subplots(nrows=2, ncols=4, figsize=(25,10), layout='constrained', subplot_kw={'projection':proj})

for i,title in enumerate(['MERRA-2', 'CTRL', r'HI$\mathbf{_{GBL}}$', r'HI$\mathbf{_{GBL}}$$-$CTRL']):
    ax[0,i].text(tx, ty, title, **text_kw)
    
#=== OBS ===
mlon = jas_mean['obs']['merra2']['h500'].lon
mlat = jas_mean['obs']['merra2']['h500'].lat
ax[0,0].pcolormesh(mlon, mlat, jas_mean['obs']['merra2']['h500'].squeeze(), cmap=cmap, norm=norm, transform=trans)
#ax[0,0].contour(mlon, mlat, jas_mean['obs']['merra2']['h500'], colors='k', levels=hlevels, linewidth=1.5, transform=trans)
ax[0,0].contour(topo['etopo'].lon, topo['etopo'].lat, topo['etopo'], levels=np.linspace(1000,4000,4), linewidths=1, colors='black', transform=trans)

ax[1,0].pcolormesh(mlon, mlat, jas_mean['obs']['merra2']['swp500'].squeeze(), cmap=dcmap, norm=dnorm, transform=trans)
ax[1,0].contour(topo['etopo'].lon, topo['etopo'].lat, topo['etopo'], levels=np.linspace(1000,4000,4), linewidths=1, colors='black', transform=trans)

#=== FLOR ===
ax[0,1].pcolormesh(lon, lat, jas_mean['pi']['ctrl']['h500'], cmap=cmap, norm=norm, transform=trans)
cf=ax[0,2].pcolormesh(lon, lat, jas_mean['pi']['hitopo']['h500'], cmap=cmap, norm=norm, transform=trans)

ax[1,1].pcolormesh(lon, lat, jas_mean['pi']['ctrl']['swp500'], cmap=dcmap, norm=dnorm, transform=trans)
cf2=ax[1,2].pcolormesh(lon, lat, jas_mean['pi']['hitopo']['swp500'], cmap=dcmap, norm=dnorm, transform=trans)

for i,run in enumerate(['ctrl','hitopo']):
    ax[0,i+1].contour(lon, lat, topo[run], levels=zlevels, linewidths=1, colors='black', transform=trans)
    ax[1,i+1].contour(lon, lat, topo[run], levels=zlevels, linewidths=1, colors='black', transform=trans)

#=== FLOR CHANGE ===
ax[0,3].text(-160,40,' ')
cf3=ax[0,3].pcolormesh(lon, lat, jas_mean['pi']['hitopo']['h500']-jas_mean['pi']['ctrl']['h500'], cmap=dcmap2, norm=mpl.colors.BoundaryNorm(np.linspace(-100, 100, 21), dcmap2.N), transform=trans)
cf4=ax[1,3].pcolormesh(lon, lat, jas_mean['pi']['hitopo']['swp500']-jas_mean['pi']['ctrl']['swp500'], cmap=dcmap2, norm=mpl.colors.BoundaryNorm(np.linspace(-20, 20, 21), dcmap2.N), transform=trans)
ax[0,3].contour(lon, lat, topo['hitopo']-topo['ctrl'], levels=np.linspace(500,1500,5), linewidths=1, colors='black', transform=trans)
ax[1,3].contour(lon, lat, topo['hitopo']-topo['ctrl'], levels=np.linspace(500,1500,5), linewidths=1, colors='black', transform=trans)

for i, ax in enumerate(ax.flat): 
    # subplot labels
    ax.text(-136, ty+1, letters[i], **text_kw2)
    # map properties
    ax.coastlines(color='k', linewidth=1.5)
    ax.set_extent(map_bnds, crs=trans)
    gl=ax.gridlines(crs=trans, lw=.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
    gl.bottom_labels=True; gl.left_labels=True; gl.top_labels=False; gl.right_labels=False
    gl.xformatter=LONGITUDE_FORMATTER
    gl.yformatter=LATITUDE_FORMATTER
    gl.xlabel_style={'color': 'black', 'weight': 'normal', 'size':14}
    gl.ylabel_style={'color': 'black', 'weight': 'normal', 'size':14}

# h500
cax=fig.add_axes([.705, .55, 0.0175, 0.4])
cbar=fig.colorbar(cf, ticks=np.linspace(5500,6000,11), orientation='vertical', extend='both', cax=cax) 
cbar.set_label('500 hPa Geopotential Height [m]', size=14, rotation=270, labelpad=20, ha='center')
cbar.ax.tick_params(labelsize=14)
# swp500
cax=fig.add_axes([.705, .05, 0.0175, 0.4])
cbar=fig.colorbar(cf2, ticks=np.linspace(-60,60,11), orientation='vertical', extend='both', cax=cax) 
cbar.set_label('Stationary Wave Pattern [m]', size=14, rotation=270, labelpad=20, ha='center')
cbar.ax.tick_params(labelsize=14)
# h500 change
cax=fig.add_axes([1, .55, 0.0174, 0.4])
cbar=fig.colorbar(cf3, ticks=np.linspace(-100,100,11), orientation='vertical', extend='both', cax=cax) 
cbar.set_label('$\Delta$ 500 hPa Geopotential Height [m]', size=14, rotation=270, labelpad=20, ha='center')
cbar.ax.tick_params(labelsize=14)
# swp500 change
cax=fig.add_axes([1, .05, 0.0175, 0.4])                                 
cbar=fig.colorbar(cf4, ticks=np.linspace(-20,20,11), orientation='vertical', extend='both', cax=cax) 
cbar.set_label('$\Delta$ Stationary Wave Pattern [m]', size=14, rotation=270, labelpad=20, ha='center')
cbar.ax.tick_params(labelsize=14)

plt.savefig(f'figs/h500.swp.flor.pdf', transparent=False, bbox_inches='tight')
plt.savefig(f'figs/h500.swp.flor.png', transparent=False, bbox_inches='tight')

In [ ]:
# -------------------- #
#       Settings       #
# -------------------- #
# plot specs
lw=1
text_kw={'color':'k', 'weight':'bold', 'size':20, 'ha':'center', 'va':'bottom'}
text_kw2={'color':'k', 'weight':'bold', 'size':26, 'ha':'center', 'va':'center'}
text_kw3={'color':'k', 'weight':'normal', 'size':14, 'ha':'left', 'va':'center'}
titles=np.array(['MERRA-2','CTRL',r'HI$\mathbf{_{GBL}}$',
                 '','',''])
letters=['A','B','C','D','E','F','G','H']
tx=-105
ty=66
# var specs
lon = jas_mean['pi']['ctrl']['h500'].lon
lat = jas_mean['pi']['ctrl']['h500'].lat
# topography contours
zlevels=np.linspace(1000,4000,8)
# h500 colormap
cmap=cm.RdYlBu_r
vmin=2900
vmax=3300
levels=np.linspace(vmin, vmax, 21)
norm=mpl.colors.BoundaryNorm(levels, cmap.N)
hlevels=np.arange(vmin-50, vmax+50, 25)
# swp500 colormap
dcmap=cm.RdBu_r
dvmin=-50
dvmax=50
dlevels=np.linspace(dvmin, dvmax, 21)
dnorm=mpl.colors.BoundaryNorm(dlevels, dcmap.N)
# diff colormap
dcmap2=cmo.balance
# map specs
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[225,290,10,65]
cols=[0,2]

fig, ax = plt.subplots(nrows=2, ncols=4, figsize=(25,10), layout='constrained', subplot_kw={'projection':proj})

for i,title in enumerate(['MERRA-2', 'CTRL', r'HI$\mathbf{_{GBL}}$', r'HI$\mathbf{_{GBL}}$$-$CTRL']):
    ax[0,i].text(tx, ty, title, **text_kw)
    
#=== OBS ===
mlon = jas_mean['obs']['merra2']['h500'].lon
mlat = jas_mean['obs']['merra2']['h500'].lat
ax[0,0].pcolormesh(mlon, mlat, jas_mean['obs']['merra2']['h700'].squeeze(), cmap=cmap, norm=norm, transform=trans)
#ax[0,0].contour(mlon, mlat, jas_mean['obs']['merra2']['h500'], colors='k', levels=hlevels, linewidth=1.5, transform=trans)
ax[0,0].contour(topo['etopo'].lon, topo['etopo'].lat, topo['etopo'], levels=np.linspace(1000,4000,4), linewidths=1, colors='black', transform=trans)

ax[1,0].pcolormesh(mlon, mlat, jas_mean['obs']['merra2']['swp700'].squeeze(), cmap=dcmap, norm=dnorm, transform=trans)
ax[1,0].contour(topo['etopo'].lon, topo['etopo'].lat, topo['etopo'], levels=np.linspace(1000,4000,4), linewidths=1, colors='black', transform=trans)

#=== FLOR ===
ax[0,1].pcolormesh(lon, lat, jas_mean['pi']['ctrl']['h700'], cmap=cmap, norm=norm, transform=trans)
cf=ax[0,2].pcolormesh(lon, lat, jas_mean['pi']['hitopo']['h700'], cmap=cmap, norm=norm, transform=trans)

ax[1,1].pcolormesh(lon, lat, jas_mean['pi']['ctrl']['swp700'], cmap=dcmap, norm=dnorm, transform=trans)
cf2=ax[1,2].pcolormesh(lon, lat, jas_mean['pi']['hitopo']['swp700'], cmap=dcmap, norm=dnorm, transform=trans)

for i,run in enumerate(['ctrl','hitopo']):
    ax[0,i+1].contour(lon, lat, topo[run], levels=zlevels, linewidths=1, colors='black', transform=trans)
    ax[1,i+1].contour(lon, lat, topo[run], levels=zlevels, linewidths=1, colors='black', transform=trans)

#=== FLOR CHANGE ===
ax[0,3].text(-160,40,' ')
cf3=ax[0,3].pcolormesh(lon, lat, jas_mean['pi']['hitopo']['h700']-jas_mean['pi']['ctrl']['h700'], cmap=dcmap2, norm=mpl.colors.BoundaryNorm(np.linspace(-100, 100, 21), dcmap2.N), transform=trans)
cf4=ax[1,3].pcolormesh(lon, lat, jas_mean['pi']['hitopo']['swp700']-jas_mean['pi']['ctrl']['swp700'], cmap=dcmap2, norm=mpl.colors.BoundaryNorm(np.linspace(-16, 16, 17), dcmap2.N), transform=trans)
ax[0,3].contour(lon, lat, topo['hitopo']-topo['ctrl'], levels=np.linspace(500,1500,5), linewidths=1, colors='black', transform=trans)
ax[1,3].contour(lon, lat, topo['hitopo']-topo['ctrl'], levels=np.linspace(500,1500,5), linewidths=1, colors='black', transform=trans)

for i, ax in enumerate(ax.flat): 
    # subplot labels
    ax.text(-136, ty+1, letters[i], **text_kw2)
    # map properties
    ax.coastlines(color='k', linewidth=1.5)
    ax.set_extent(map_bnds, crs=trans)
    gl=ax.gridlines(crs=trans, lw=.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
    gl.bottom_labels=True; gl.left_labels=True; gl.top_labels=False; gl.right_labels=False
    gl.xformatter=LONGITUDE_FORMATTER
    gl.yformatter=LATITUDE_FORMATTER
    gl.xlabel_style={'color': 'black', 'weight': 'normal', 'size':14}
    gl.ylabel_style={'color': 'black', 'weight': 'normal', 'size':14}

# h500
cax=fig.add_axes([.705, .55, 0.0175, 0.4])
cbar=fig.colorbar(cf, ticks=np.linspace(2900,3300,9), orientation='vertical', extend='both', cax=cax) 
cbar.set_label('700 hPa Geopotential Height [m]', size=14, rotation=270, labelpad=20, ha='center')
cbar.ax.tick_params(labelsize=14)
# swp500
cax=fig.add_axes([.705, .05, 0.0175, 0.4])
cbar=fig.colorbar(cf2, ticks=np.linspace(-50,50,11), orientation='vertical', extend='both', cax=cax) 
cbar.set_label('Stationary Wave Pattern [m]', size=14, rotation=270, labelpad=20, ha='center')
cbar.ax.tick_params(labelsize=14)
# h500 change
cax=fig.add_axes([1, .55, 0.0174, 0.4])
cbar=fig.colorbar(cf3, ticks=np.linspace(-100,100,11), orientation='vertical', extend='both', cax=cax) 
cbar.set_label('$\Delta$ 700 hPa Geopotential Height [m]', size=14, rotation=270, labelpad=20, ha='center')
cbar.ax.tick_params(labelsize=14)
# swp500 change
cax=fig.add_axes([1, .05, 0.0175, 0.4])                                 
cbar=fig.colorbar(cf4, ticks=np.linspace(-16,16,9), orientation='vertical', extend='both', cax=cax) 
cbar.set_label('$\Delta$ Stationary Wave Pattern [m]', size=14, rotation=270, labelpad=20, ha='center')
cbar.ax.tick_params(labelsize=14)

plt.savefig(f'figs/h700.swp.flor.pdf', transparent=False, bbox_inches='tight')
plt.savefig(f'figs/h700.swp.flor.png', transparent=False, bbox_inches='tight')

### h500 climo and bias for pi and 2xco2

In [ ]:
# -------------------- #
#       Settings       #
# -------------------- #
# plot specs
lw=1
text_kw={'color':'k', 'weight':'bold', 'size':20, 'ha':'center', 'va':'bottom'}
text_kw2={'color':'k', 'weight':'bold', 'size':26, 'ha':'center', 'va':'center'}
text_kw3={'color':'k', 'weight':'normal', 'size':14, 'ha':'left', 'va':'center'}
titles=np.array(['MERRA2','CM2.5-FLOR CTRL','CM2.5-FLOR HI-GBL',
                 '','',''])
letters=['A','B','B','B','B','C','D','E','F','G','H','I','J']
tx=-105
ty=66
# var specs
lon = jas_mean['flor']['ctrl']['h500'].lon
lat = jas_mean['flor']['ctrl']['h500'].lat
# topography contours
zlevels=np.linspace(1000,4000,8)
# climo colormap
cmap=cm.RdYlBu_r
vmin=5500
vmax=6000
levels=np.linspace(vmin, vmax, 21)
norm=mpl.colors.BoundaryNorm(levels, cmap.N)
hlevels=np.arange(vmin-50, vmax+50, 25)
# bias colormap
dcmap=cm.RdBu_r
dvmin=-100
dvmax=100
dlevels=np.linspace(dvmin, dvmax, 21)
dnorm=mpl.colors.BoundaryNorm(dlevels, dcmap.N)
# map specs
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[225,290,10,65]
cols=[0,2]

fig, ax = plt.subplots(nrows=3, ncols=4, figsize=(20,13), layout='constrained', subplot_kw={'projection':proj})

ax[0,0].text(-147,32.5,'MERRA2', rotation=90, **text_kw)
ax[1,0].text(-147,32.5,'CTRL', rotation=90, **text_kw)
ax[2,0].text(-147,32.5,r'HI$\mathbf{_{GBL}}$', rotation=90, **text_kw)
for i,title in enumerate(['PI', 'MERRA2$-$PI', '2$x$CO$_2$', 'MERRA2$-$2$x$CO$_2$']):
    ax[1,i].text(tx, ty, title, **text_kw)
    
#=== OBS ===
mlon = jas_mean['obs']['merra2']['h500'].lon
mlat = jas_mean['obs']['merra2']['h500'].lat
ax[0,0].pcolormesh(mlon, mlat, jas_mean['obs']['merra2']['h500'], cmap=cmap, norm=norm, transform=trans)
#ax[0,0].contour(mlon, mlat, jas_mean['obs']['merra2']['h500'], colors='k', levels=hlevels, linewidth=1.5, transform=trans)
ax[0,0].contour(topo['etopo'].lon, topo['etopo'].lat, topo['etopo'], levels=np.linspace(1000,4000,4), linewidths=.75, colors='black', transform=trans)

ax[0,1].set_visible(False)
ax[0,2].set_visible(False)
ax[0,3].set_visible(False)

#=== FLOR ===
run='ctrl'
i=0
for key in ['flor','2xco2']:
    cf=ax[1,i].pcolormesh(lon, lat, jas_mean[key][run]['h500'], cmap=cmap, norm=norm, transform=trans)
    cf2=ax[1,i+1].pcolormesh(lon, lat, obs_diff_mask[key][run]['h500'], cmap=dcmap, norm=dnorm, transform=trans)
    #ax[1,i+1].contourf(lon, lat, bias_change_masked[key][run]['h500'],
    #                   0, colors='none', hatches=['','///'], extend='lower', zorder=100, transform=trans)
    i+=2

run='hitopo'
i=0
for key in ['flor','2xco2']:
    ax[2,i].pcolormesh(lon, lat, jas_mean[key][run]['h500'], cmap=cmap, norm=norm, transform=trans)
    ax[2,i+1].pcolormesh(lon, lat, obs_diff_mask[key][run]['h500'], cmap=dcmap, norm=dnorm, transform=trans)
    ax[2,i+1].contourf(lon, lat, bias_change_masked[key][run]['h500'],
                       0, colors='none', hatches=['','///'], extend='lower', zorder=100, transform=trans)
    i+=2           

for i,run in enumerate(['ctrl','hitopo']):
    for j in [0,1,2,3]:
        ax[i+1,j].contour(lon, lat, topo[run], levels=zlevels, linewidths=.75, colors='black', transform=trans)

for i, ax in enumerate(ax.flat): 
    # subplot labels
    ax.text(-136, ty+1, letters[i], **text_kw2)
    # map properties
    ax.coastlines(color='k', linewidth=1)
    ax.set_extent(map_bnds, crs=trans)
    gl=ax.gridlines(crs=trans, lw=.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
    gl.bottom_labels=True; gl.left_labels=True; gl.top_labels=False; gl.right_labels=False
    gl.xformatter=LONGITUDE_FORMATTER
    gl.yformatter=LATITUDE_FORMATTER
    gl.xlabel_style={'color': 'black', 'weight': 'normal', 'size':14}
    gl.ylabel_style={'color': 'black', 'weight': 'normal', 'size':14}

# climo
cax=fig.add_axes([.3, 0.85, 0.65, 0.025])
cbar=fig.colorbar(cf, ticks=np.linspace(5500,6000,11), orientation='horizontal', extend='both', cax=cax) 
cbar.set_label('h500 [m]', labelpad=-65, size=18, fontweight='bold', ha='center')
cbar.ax.tick_params(labelsize=18)

# bias
cax=fig.add_axes([.3, 0.725, 0.65, 0.025])
cbar=fig.colorbar(cf2, ticks=np.linspace(-100,100,11), orientation='horizontal', extend='both', cax=cax) 
cbar.set_label(r'$\mathbf{\Delta}$h500 [m]', labelpad=-65, size=18, fontweight='bold', ha='center')
cbar.ax.tick_params(labelsize=18)

plt.savefig(f'figs/h500.flor.pi.2xco2.bias.change.pdf', transparent=False, bbox_inches='tight')
plt.savefig(f'figs/h500.flor.pi.2xco2.bias.change.pdf', transparent=False, bbox_inches='tight')

### CTRL & HIGBL vs. MERRA2 for PI only

In [ ]:
# -------------------- #
#       Settings       #
# -------------------- #
# plot specs
lw=1
text_kw={'color':'k', 'weight':'bold', 'size':16, 'ha':'center', 'va':'bottom'}
text_kw2={'color':'k', 'weight':'bold', 'size':20, 'ha':'center', 'va':'center'}
text_kw3={'color':'k', 'weight':'normal', 'size':10, 'ha':'left', 'va':'center'}
titles=np.array(['MERRA2 $-$ CM2.5-FLOR CTRL','MERRA2 $-$ CM2.5-FLOR HI-GBL','CM2.5-FLOR HI-GBL $-$ CTRL'])
letters=['A','B','C','D']
tx=-107.5
ty=56
# var specs
lon = jas_mean['flor']['ctrl']['h500'].lon
lat = jas_mean['flor']['ctrl']['h500'].lat
# topography contours
zlevels=np.linspace(1000,3000,7)
# climo colormap
cmap=cm.RdYlBu_r
vmin=5500
vmax=6000
levels=np.linspace(vmin, vmax, 21)
norm=mpl.colors.BoundaryNorm(levels, cmap.N)
hlevels=np.arange(vmin-50, vmax+50, 25)
# bias colormap
dcmap=cm.RdBu_r
dvmin=-100
dvmax=100
dlevels=np.linspace(dvmin, dvmax, 21)
dnorm=mpl.colors.BoundaryNorm(dlevels, dcmap.N)
# map specs
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[225,280,10,55]
cols=[0,2]


# ------------------- #
#      Make Plot      #
# ------------------- #
fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(17,5.5), layout='constrained', subplot_kw={'projection':proj})

#fig.text(.5,1.0,f'Modified Topo $-$ CTRL', **text_kw)
fig.text(0.025,0,
         'Masked for statistical significance. Hatching indicates where higher topography increased model biases (calculated as MERRA2$-$Model).\n'
         'Thin black lines are model surface height boundary conditions, intervals of 500 m starting at 1 km.',
         **text_kw3)

#=== CTRL ===
for i,run in enumerate(['ctrl','hitopo']):
    cf=ax[i].pcolormesh(lon, lat, obs_diff_mask['flor'][run]['h500'], cmap=dcmap, norm=dnorm, transform=trans)
    ax[i].contour(lon, lat, topo[run], levels=zlevels, linewidths=.8, colors='black', transform=trans)

#=== HI-GBL ===
ax[2].pcolormesh(lon, lat, model_diff_mask['flor']['hitopo']['h500'], cmap=dcmap, norm=dnorm, transform=trans)
ax[2].contour(lon, lat, topo['hitopo'], levels=zlevels, linewidths=.8, colors='black', transform=trans)
# add hatching where bias got worse
ax[2].contourf(lon, lat, bias_change_masked['flor']['hitopo']['h500'],
               0, colors='none', hatches=['','///'], extend='lower', zorder=100, transform=trans)

for i, ax in enumerate(ax.flat): 
    # subplot labels
    ax.text(tx, ty, titles[i], **text_kw)
    ax.text(-136, ty+1, letters[i], **text_kw2)
    # map properties
    ax.coastlines(color='k', linewidth=1.5)
    ax.add_feature(cfeature.BORDERS, edgecolor='k', linewidth=1.5)
    ax.set_extent(map_bnds, crs=trans)
    gl=ax.gridlines(crs=trans, lw=.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
    gl.bottom_labels=True; gl.left_labels=True; gl.top_labels=False; gl.right_labels=False
    gl.xformatter=LONGITUDE_FORMATTER
    gl.yformatter=LATITUDE_FORMATTER
    gl.xlabel_style={'color': 'black', 'weight': 'normal', 'size':12}
    gl.ylabel_style={'color': 'black', 'weight': 'normal', 'size':12}

# colorbar
cax=fig.add_axes([1.01, 0.15, 0.02, 0.7])
cbar=fig.colorbar(cf, ticks=np.linspace(-100,100,5), orientation='vertical', extend='both', cax=cax)
cbar.set_label('$\Delta$ h500 [m]', labelpad=8, rotation=270, size=14, fontweight='normal', ha='center')
cbar.ax.tick_params(labelsize=14)
for tick in cbar.ax.yaxis.get_major_ticks():
    tick.label2.set_fontweight('normal')

#plt.savefig(f'{opath}/slp.flor.e3.nam-change.{season}.pdf', transparent=False, bbox_inches='tight')

### h500 climatology for FLOR PI and 2xCO2

In [ ]:
# -------------------- #
#       Settings       #
# -------------------- #
# plot specs
lw=1
text_kw={'color':'k', 'weight':'bold', 'size':20, 'ha':'center', 'va':'bottom'}
text_kw2={'color':'k', 'weight':'bold', 'size':26, 'ha':'center', 'va':'center'}
text_kw3={'color':'k', 'weight':'normal', 'size':14, 'ha':'left', 'va':'center'}
titles=np.array(['MERRA2','CM2.5-FLOR CTRL','CM2.5-FLOR HI-GBL',
                 '','',''])
letters=['A','B','C','D','D','E','F']
tx=-105
ty=66
# var specs
lon = jas_mean['flor']['ctrl']['h500'].lon
lat = jas_mean['flor']['ctrl']['h500'].lat
# topography contours
zlevels=np.linspace(1000,3000,7)
# climo colormap
cmap=cm.RdYlBu_r
vmin=5500
vmax=6000
levels=np.linspace(vmin, vmax, 21)
norm=mpl.colors.BoundaryNorm(levels, cmap.N)
hlevels=np.arange(vmin-50, vmax+50, 25)
# bias colormap
dcmap=cm.RdBu_r
dvmin=-100
dvmax=100
dlevels=np.linspace(dvmin, dvmax, 21)
dnorm=mpl.colors.BoundaryNorm(dlevels, dcmap.N)
# map specs
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[225,290,10,65]
cols=[0,2]


# ------------------- #
#      Make Plot      #
# ------------------- #
fig, ax = plt.subplots(nrows=2, ncols=3, figsize=(16,10), layout='constrained', subplot_kw={'projection':proj})

#fig.text(.5,1.0,f'Modified Topo $-$ CTRL', **text_kw)
fig.text(0.025,0,
         'Thin black lines are model surface height boundary conditions, intervals of 500 m starting at 1 km.',
         **text_kw3)

#=== OBS ===
ax[0,0].text(-145,32.5,'PI/Modern', rotation=90, **text_kw)
mlon = jas_mean['obs']['merra2']['h500'].lon
mlat = jas_mean['obs']['merra2']['h500'].lat
cf=ax[0,0].pcolormesh(mlon, mlat, jas_mean['obs']['merra2']['h500'], cmap=cmap, norm=norm, transform=trans)
ax[0,0].contour(mlon, mlat, jas_mean['obs']['merra2']['h500'], colors='k', levels=hlevels, linewidth=1.5, transform=trans)
ax[0,0].contour(topo['etopo'].lon, topo['etopo'].lat, topo['etopo'], levels=zlevels, linewidths=.75, colors='black', transform=trans)
#=== CTRL ===
for i,key in enumerate(['flor','2xco2']):
    for j,run in enumerate(['ctrl','hitopo']):
        j=j+1
        ax[i,j].pcolormesh(lon, lat, jas_mean[key][run]['h500'], cmap=cmap, norm=norm, transform=trans)
        ax[i,j].contour(lon, lat, jas_mean[key][run]['h500'], colors='k', levels=hlevels, linewidth=0.75, transform=trans)
        ax[i,j].contour(lon, lat, topo[run], levels=zlevels, linewidths=.75, colors='black', transform=trans)
         
ax[1,0].set_visible(False)

for i, ax in enumerate(ax.flat): 
    # subplot labels
    ax.text(tx, ty, titles[i], **text_kw)
    ax.text(-136, ty+1, letters[i], **text_kw2)
    # map properties
    ax.coastlines(color='k', linewidth=1)
    ax.set_extent(map_bnds, crs=trans)
    gl=ax.gridlines(crs=trans, lw=.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
    gl.bottom_labels=True; gl.left_labels=True; gl.top_labels=False; gl.right_labels=False
    gl.xformatter=LONGITUDE_FORMATTER
    gl.yformatter=LATITUDE_FORMATTER
    gl.xlabel_style={'color': 'black', 'weight': 'normal', 'size':14}
    gl.ylabel_style={'color': 'black', 'weight': 'normal', 'size':14}

# colorbar
cax=fig.add_axes([1.01, 0.15, 0.025, 0.7])
cbar=fig.colorbar(cf, ticks=np.linspace(5500,6000,11), orientation='vertical', extend='both', cax=cax) 
cbar.set_label('h500 [m]', labelpad=25, rotation=270, size=18, fontweight='normal', ha='center')
cbar.ax.tick_params(labelsize=18)
for tick in cbar.ax.yaxis.get_major_ticks():
    tick.label2.set_fontweight('normal')

#plt.savefig(f'{opath}/slp.flor.e3.nam-change.{season}.pdf', transparent=False, bbox_inches='tight')